### Exploration
In this notebook, we'll explore the current limitations of tiny language models for mathematical reasoning. We'll start by experimenting with a low (billions) parameter model, running it on a variety of math questions to better understand its level of mathematical ability, discover its weaknesses, and analyze where it succeeds and where it fails.

#### Loading the model

In [4]:
# Setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from utils import models
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")
model_str = "Qwen/Qwen3-1.7B-Base"  # base model

tokenizer = models.load_tokenizer(model_str)
model = models.load_model(model_str)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 610358.23 examples/s]


FileNotFoundError: Directory '/nobackup/davidhellstrom' does not exist.
You have to create it before continuing!

### Below is the format used in the [**DeepSeek R1 paper**](https://arxiv.org/abs/2501.12948)

<div style="font-size: 0.85em; max-width: 750px; line-height: 1.5;">

---
*A conversation between User and Assistant. The user asks a question, and the Assistant solves 
it. The assistant first thinks about the reasoning process in the mind and then provides the user 
with the answer. The reasoning process and answer are enclosed within \<think>...\</think> 
and \<answer>...\</answer> tags, respectively, i.e., \<think> reasoning process here \</think> 
\<answer> answer here \</answer>. User: <span style="color:red">prompt</span>. Assistant:*

---

</div>

We are working with a base model, which is why we have to specify that this is a conversation between a User and Assistant, as well as have the ending be **Assistant:**, because the model will try to predict what an assistant would respond in such a scenario. 

We can create a function for generating this "system prompt" for an arbitrary question, by replacing the <span style="color:red">prompt</span>.

In [ ]:
def generate_prompt(question, helper="", think_start_tok="<think>", think_stop_tok="</think>", answer_start_tok="<answer>", answer_stop_tok="</answer>"):
    """
    Wraps a question into the DeepSeek-R1 paper prompt format.
    """

    prompt = (
        "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
        "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
        f"The reasoning process and answer are enclosed within {think_start_tok}...{think_stop_tok} and {answer_start_tok}...{answer_stop_tok} tags, "
        f"respectively, i.e., {think_start_tok} reasoning process here {think_stop_tok} {answer_start_tok} answer here {answer_stop_tok}. "
        f"User: {question} Assistant: {helper}"
    )
    return prompt

We can now use the model with the "system" prompt, and try to get it to answer a math question.

Most likely, the model will completely fail at using the specified tags like it should, even with the added `\<think>` helper token we added.

In [ ]:
from utils import lmprint

question = "what is 14 times 5670?"

inputs = tokenizer(generate_prompt(question, "<think>"), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

output = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=0.8,        # Adds randomness
                top_p=0.9,              
                pad_token_id=tokenizer.eos_token_id
            )

generated_tokens = output[0][input_len:]

raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
raw_full = tokenizer.decode(output[0], skip_special_tokens=False)
lmprint.pretty_print("<think>" + raw_output)
print(f"Raw output: {raw_full}")

### Why this happens:
Since we are using a base model, this means it has **only** been pretrained.
Our tokenizer has many added tokens, which the base model has never seen, for example `<think>` and `</think>`.
This means that the embeddings for those new tokens are just random and dont have any meaning. 
This is not the case thought for `<answer>`, which is why it manages to output it correctly (most of the time)

In [ ]:
# Check if these are single tokens or get split into sub-tokens
test_strings = ["<think>", "</think>", "<answer>", "</answer>"]

for s in test_strings:
    tokens = tokenizer.tokenize(s)
    ids = tokenizer.encode(s, add_special_tokens=False)
    print(f"{s:<15} -> tokens: {tokens}, ids: {ids}")


If the model cannot even follow along with the tags that we specified, perhaps it will not be possible to even get it to start improving, because it will always get bad rewards, and never be able to start improving.

Lets try to give our model some chances at this, because it only has to do it a couple of times for each rollout, since we are going to be doing multiple generations per question. 
Eventually it should learn to do this very consistently.

In [ ]:
from utils import checks
from rewards import rewards

def print_response(model, question, temperature):
    inputs = tokenizer(generate_prompt(question), return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[1]

    outputs = model.generate(
                    **inputs, 
                    max_new_tokens=256,    # High, but just a demonstration
                    do_sample=True,         # Enable sampling
                    temperature=temperature,        # Adds randomness
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )

    output_texts = tokenizer.decode(outputs[0, input_len:], skip_special_tokens=True)
    lmprint.print_question(question)
    lmprint.pretty_print(output_texts)

def generate_responses(model, question, num_responses, temperature):
    inputs = tokenizer(generate_prompt(question), return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[1]

    outputs = model.generate(
                    **inputs, 
                    max_new_tokens=256,    # High, but just a demonstration
                    do_sample=True,         # Enable sampling
                    temperature=temperature,        # Adds randomness
                    num_return_sequences=num_responses, # This does the parallel work for n tries
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )

    output_texts = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    return output_texts

def evaluate(model, questions, answers=[], G=1, temperature = 1.0):
    num_questions = len(questions)

    answer_count = 0
    think_count = 0
    perfect_count = 0
    correct_count = 0

    for q_idx in range(num_questions):
        question = questions[q_idx]
        responses = generate_responses(model, question, G, temperature)
        for raw_response in responses:
            if checks.has_complete_answer_block(raw_response): answer_count += 1
            if checks.has_complete_thinking_block(raw_response): think_count += 1
            if checks.is_format_correct(raw_response): perfect_count += 1
            if answers:
                ground_truth = answers[q_idx]
                if checks.is_correct_answer(raw_response, ground_truth): correct_count += 1

    num_samples = num_questions * G
    print(f"Thinking: {(think_count/num_samples)*100:.1f}% of tries.\n",
        f"Answer: {(answer_count/num_samples)*100:.1f}% of tries\n",
        f"Perfect format: {(perfect_count/num_samples)*100:.1f}% of tries")
    if answers:
        print(f"Correct answers: {(correct_count/num_samples)*100:.1f}% of tries")
    
questions = ["What is 11 * 9?", 
             "Sam has 4 apples, and eats 2 of them, and gives one to Anna, how many apples does he have left?",
             "Solve: (5 + 1) * 21"]
evaluate(model=model, questions=questions, G=8)

---

So our model uses the correct thinking tokens 0% of all the generated outputs, so we wont ever be actually learning the correct formatting if this is the case.

--- 

### Possible fixes.
1. Average the new token from the subcomponents that make it up.
    * Set the token `<think>` to the average of the tokens `<`, `think`, `>`.
    * Set the token `</think>` to the average of the tokens `</`, `think`, `>`.
    * This did not work well enough to actually start generating full `<think>` tokens, but instead just closed the thinking with `>`
2. Tiny SFT dataset with formatting data
    * Train model on generated examples where the model uses the think and answer tokens.
    * Ex: "User: What is 2*13? Assistant: \<think>2\*13=26\</think>\<answer>26\</answer>


In [ ]:
from data.generate import RandomFormatSFTDataset

SFTdataset = RandomFormatSFTDataset(tokenizer=tokenizer, prompt_template=generate_prompt, size=200)
print("Input text:")
print(tokenizer.decode(SFTdataset[0]["input_ids"]))

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch

# TRAINING CONFIG
EPOCHS      = 5
BATCH_SIZE  = 8
LR          = 5e-4

# FREEZE MODEL PARAMETERS
for param in model.parameters():
    param.requires_grad = False

# UNFREEZE EMBEDDING AND HEAD LAYER
model.get_input_embeddings().weight.requires_grad = True
model.lm_head.weight.requires_grad = True
trainable_params = [param for param in model.parameters() if param.requires_grad]
print(f"Trainable parameters: {sum(p.numel() for p in trainable_params)}")

optimizer = AdamW(trainable_params, lr=LR)

# Pad a training batch so that everyting is same length
def collate_fn(batch):
    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id
    # Append EOS to each sample
    new_batch = []
    for b in batch:
        new_batch.append({
            "input_ids": torch.cat([b["input_ids"], torch.tensor([eos_id])]),
            "labels": torch.cat([b["labels"], torch.tensor([eos_id])]),
            "attention_mask": torch.cat([b["attention_mask"], torch.ones(1)]),
        })
    
    max_len = max(b["input_ids"].shape[0] for b in new_batch)
    
    input_ids = []
    labels = []
    attention_mask = []
    
    for b in new_batch:
        pad_len = max_len - b["input_ids"].shape[0]
        input_ids.append(torch.cat([b["input_ids"], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        labels.append(torch.cat([b["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))
        attention_mask.append(torch.cat([b["attention_mask"], torch.zeros(pad_len)]))
    
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attention_mask),
    }

dataloader = DataLoader(SFTdataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

# TRAINING LOOP
model.train()
step = 0

for epoch in range(EPOCHS):
    epoch_loss = 0
    num_batches = 0

    for batch in dataloader:
        # Move to device
        batch = {k: v.to(model.device) for k, v in batch.items()}

        # Forward
        outputs = model(**batch)
        loss = outputs.loss

        # Backward
        loss.backward()

        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=2.0)


        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        num_batches += 1
        step += 1

        if step % 10 == 0:
            print(f"  Step {step:>4d} | Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / num_batches
    print(f"Epoch {epoch+1:>2d}/{EPOCHS} | Avg Loss: {avg_loss:.4f}")

# Set model to eval mode again, and unfreeze params
for param in model.parameters():
    param.requires_grad = True
model.eval()


In [ ]:
from utils.models import save_model
save_model(model, model_str+"-sft")

### After training
We have now trained our model on some examples where the model makes use of the thinking tags. We can now test if it has learned to use the new token.

In [ ]:
questions = ["What is 11 * 9?", 
             "Sam has 4 apples, and eats 2 of them, and gives one to Anna, how many apples does he have left?",
             "Solve: (5 + 1) * 21",
             "What is the next number for the sequence [1,2,4,8,16]?"]
answers = ["99", "1", "126", "32"]
evaluate(model=model, questions=questions, answers=answers, G=16, temperature = 1.0)

In [ ]:
print_response(model, questions[3], temperature = 1.0)

# Calculate rewards 
Section showing how the rewards are calculated, with a few examples.

In [ ]:
"""
Calculating average rewards for n rounds of output for different math questions 
with given answers using our model.
"""

from rewards import rewards

questions = ["What is (100 - 12) * 2 - 40?", "What is 50 * 5 + 60?", "What is (100 + 100) * (2 - 40)?"]
answers = ["136", "310", "-7600"]
number_of_rounds = 5

for index, question in enumerate(questions):
    total_reward = 0
    for round in range(1,number_of_rounds+1):
        inputs = tokenizer(generate_prompt(question), return_tensors="pt").to(model.device)
        input_len = inputs.input_ids.shape[1]
        output = model.generate(
                        **inputs, 
                        max_new_tokens=256,    # High, but just a demonstration
                        do_sample=True,         # Enable sampling
                        temperature=2.0,        # Adds randomness
                        top_p=0.9,              
                        pad_token_id=tokenizer.eos_token_id
                    )
        generated_tokens = output[0][input_len:]
        raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=False)
        raw_full = tokenizer.decode(output[0], skip_special_tokens=False)
        #print("raw_output: ", raw_output)
        #print("raw_full: ", raw_full)
        #lmprint.pretty_print(raw_output)
        #print(f"Raw output for question {index}: {raw_full}")

        #print(f"Testcase {index}:")
        #print(f"Reward (correct answer): {rewards.calculate_reward(raw_output, answers[index]):.2f}")
        total_reward += rewards.calculate_reward(raw_output, answers[index])

    average_reward = total_reward/(number_of_rounds)

    print(f"Average reward for {number_of_rounds} rounds with Question {index} is: {average_reward} ")


"""
In test_rewards.py we are checking that the reward function works by testing
all edge cases for the calculate_reward() function.
"""



Now that we have all the prerequisites done, we can start exploring the **GRPO** algorithm. 

The full algorithm from the R1 paper looks as follows:
$$\mathcal{L}^{\text{GRPO}}(\theta) = \mathbb{E}_{q \sim P(Q),\; \{o_i\}_{i=1}^{G} \sim \pi_{\theta_{\text{old}}}(\cdot \mid q)} \left[ \frac{1}{G} \sum_{i=1}^{G} \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \left( \min\!\Big( r_{i,t}(\theta)\, \hat{A}_i,\; \text{clip}\!\big(r_{i,t}(\theta),\, 1-\epsilon,\, 1+\epsilon\big)\, \hat{A}_i \Big) - \beta\, D_{\text{KL}}\!\big(\pi_{\theta} \,\|\, \pi_{\text{ref}}\big) \right) \right]$$

### Calculating the ratio
$$r_t(\theta) = \frac{\pi_{\theta}(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)} = \exp\!\Big(\log \pi_{\theta}(a_t \mid s_t) - \log \pi_{\theta_{\text{old}}}(a_t \mid s_t)\Big)$$

First, we calculate the actual log probabilites for each token.

In [ ]:
import torch.nn.functional as F
from transformers import PreTrainedModel

def get_per_token_logps(model: PreTrainedModel, input_ids: torch.LongTensor, attention_mask: torch.LongTensor) -> torch.FloatTensor:
    """Returns the log probabilities for generating each output token, based on the previous tokens"""
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    # Logits is shape [G, SequenceLength, VocabSize]

    # Logits [:, t, :] predicts tokens t+1
    # So we only need up to logits[:, t-1, :]
    log_probs = torch.log_softmax(logits[:, :-1, :], dim=-1)
    # log_probs is shape [G, SequenceLength-1, VocabSize]

    # We need only the log probability for the model predicting the actual generated token.
    # So we index the vocab dimension with our input_ids.
    # input_ids is shape [G, SequenceLength] -> want shape [G, SequenceLength-1, 1]
    token_logps = log_probs.gather(
        dim=-1, 
        index=input_ids[:, 1:].unsqueeze(dim=-1),
    ).squeeze(dim=-1) # Shape [G, SequenceLength-1]

    token_logps = F.pad(token_logps, pad=(1, 0), value=0.0)

    return token_logps

def compute_ratio(model_new, model_old, input_ids, attention_mask, response_mask):
    """
    Calculate probability ratio between two models, π / π_old.
    Simplified as exp(log(π) - log(π_old)).

    How likely is our new model to produce this output, compared to the old model.
    We dont want to stray too far from the old model.
    """
 
    model_new_logps = get_per_token_logps(model_new, input_ids, attention_mask)

    with torch.no_grad():
        model_old_logps = get_per_token_logps(model_old, input_ids, attention_mask)

    # Our probabilities are now in log-space, so log(π/π_old) = log(π) - log(π_old)
    log_ratio = model_new_logps - model_old_logps

    # Zero out the prompt tokens, because we dont train on those
    log_ratio_response = log_ratio * response_mask

    # Now we can exponentiate to get back to the actual ratio
    ratio = torch.exp(log_ratio_response)

    return ratio

In [ ]:
import torch

old_model = AutoModelForCausalLM.from_pretrained(model_str, dtype="auto", device_map="auto")

prompt = (
    "A conversation between User and Assistant. The user asks a question, "
    "and the Assistant solves it. The reasoning process and answer are enclosed "
    "within <think>...</think> and <answer>...</answer> tags.\n"
    "User: What is (100 - 12) * 2 - 40?\nAssistant:"
)

response = (
    " <think>\n100 - 12 = 88. Then: 88 * 2 = 176. 176 - 40 = 136.\n"
    "</think>\n<answer>136</answer>"
)

# Tokenize separately → exact boundary, no offset mapping needed
prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
response_ids = tokenizer.encode(response, add_special_tokens=False)
prompt_len = len(prompt_ids)

full_ids = torch.tensor([prompt_ids + response_ids], device=model.device)
attention_mask = torch.ones_like(full_ids, dtype=torch.float)
response_mask = torch.zeros_like(full_ids, dtype=torch.float)
response_mask[:, prompt_len:] = 1.0

# Compute ratios
ratio = compute_ratio(model, old_model, full_ids, attention_mask, response_mask)

# ── Display: show response as annotated text ──
response_ratios = ratio[0, prompt_len:].detach().cpu()

print("Response with ratios:\n")
for i, token_id in enumerate(response_ids):
    tok = repr(tokenizer.decode([token_id]))
    r = response_ratios[i].item()
    bar = "█" * int(min(abs(r - 1.0) * 50, 20))  # visual scale
    direction = "↑" if r > 1.0 else "↓" if r < 1.0 else "="
    print(f"  {tok:<15s}  {r:.4f} {direction} {bar}")

print(f"\n{'─' * 50}")
print(f"  Mean:  {response_ratios.mean():.4f}")
print(f"  Range: [{response_ratios.min():.4f}, {response_ratios.max():.4f}]")

### Calculate clipped surrogate objective
Now that we have calculated the ratio, lets move on to calculting the full clipped surrogate objective:

$$
L_{i,t}^{\text{clip}}(\theta) = \min\!\Big( r_{i,t}(\theta)\, \hat{A}_i,\;\; \text{clip}\!\big(r_{i,t}(\theta),\; 1-\epsilon,\; 1+\epsilon\big)\, \hat{A}_i \Big)
$$

We can begin by computing the advantage:
$$
\hat{A}_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})}
$$
Where $\mathbf{r} = \{r_1, r_2, \dots, r_G\}$ are the rewards for all $G$ outputs generated for the same question.



In [5]:
def compute_advantages(rewards: torch.Tensor) -> torch.Tensor:
    """
    Compute advantages
    """
    return (rewards - rewards.mean()) / (rewards.std() + 1e-8) # Add small epsilon to prevent division by zero

And then computing the **clipped surrogate objective**:

In [ ]:
def clipped_surrogate_objective(ratios, advantages, epsilon):
    unclipped = ratios * advantages
    clipped = torch.clip(ratios, 1-epsilon, 1+epsilon) * advantages
    return torch.min(unclipped, clipped)


### KL Divergence
Now for the final component, the **KL Divergence**:
$$
D_{\text{KL}}\big(\pi_{\theta} \,\|\, \pi_{\text{ref}}\big) = \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta}(o_{i,t} \mid q, o_{i,<t})} - \log \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta}(o_{i,t} \mid q, o_{i,<t})} - 1
$$

The job of the KL Divergence is to not have the model diverge too much from the original model.
This might not even be necessary for the sake of our small scale experiment, so we can try both with and without it too see which one works best for us.

In [ ]:
def compute_kl_penalty(model_new, model_ref, input_ids, attention_mask, response_mask):
    """Per-token KL divergence estimate, only on response tokens."""
    new_logps = get_per_token_logps(model_new, input_ids, attention_mask)
    
    with torch.no_grad():
        ref_logps = get_per_token_logps(model_ref, input_ids, attention_mask)
    
    log_ratio = ref_logps - new_logps
    kl = torch.exp(log_ratio) - log_ratio - 1
    
    # Zero out prompt tokens
    kl = kl * response_mask
    
    return kl

### GRPO Loss
We can now implement the full GRPO Loss using these functions

In [ ]:
def grpo_loss(ratios, advantages, response_mask, epsilon=0.2):
    """
    L = 1/G Σ 1/|o_i| Σ min(r * A, clip(r) * A)
    """
    adv = advantages.unsqueeze(1)                                       # [G] → [G, 1]
    surrogate = clipped_surrogate_objective(ratios, adv, epsilon)       # [G, seq_len]

    token_counts = response_mask.sum(dim=1).clamp(min=1)                # [G] = |o_i|
    per_output = (surrogate * response_mask).sum(dim=1) / token_counts  # 1/|o_i| Σ_t

    return -per_output.mean()     

def grpo_loss_with_kl(ratios, advantages, kl, response_mask, epsilon=0.2, beta=0.01):
    """
    L = 1/G Σ 1/|o_i| Σ (min(r * A, clip(r) * A) - β * KL)
    """
    adv = advantages.unsqueeze(1)                                                # [G] → [G, 1]
    surrogate = clipped_surrogate_objective(ratios, adv, epsilon)                # [G, seq_len]

    penalized = surrogate - beta * kl                                            # [G, seq_len]

    token_counts = response_mask.sum(dim=1).clamp(min=1)                         # [G] = |o_i|
    per_output = (penalized * response_mask).sum(dim=1) / token_counts           # 1/|o_i| Σ_t

    return -per_output.mean()                                                    # -1/G Σ_i                                      # -1/G Σ_i

### GSM8K maj@16 test

We evaluate on GSM8K **test** with **majority vote over 16 sampled answers (maj@16)**.

For each question $q_i$ with gold integer answer $y_i$:
1. Create prompt $p_i=\texttt{generate\_prompt}(q_i)$.
2. Sample $G=16$ completions $c_{i,1},\dots,c_{i,16} \sim \pi_\theta(\cdot\mid p_i)$ (e.g. temperature/top-$p$).
3. Extract an integer from each completion: $\hat{y}_{i,j}=f(c_{i,j})$ (prefer `<answer>...</answer>`, fallback: last integer).
4. Predict by majority vote:
$
\tilde{y}_i=\arg\max_{v}\sum_{j=1}^{16}\mathbf{1}\{\hat{y}_{i,j}=v\}.
$
5. Report exact-match accuracy:
$
\text{Acc}_{\text{maj@16}}=\frac{1}{N}\sum_{i=1}^{N}\mathbf{1}\{\tilde{y}_i=y_i\}.
$

(Optional) **pass@16** counts an item correct if any sample matches:  
$\mathbf{1}\{\exists j:\hat{y}_{i,j}=y_i\}$.

In [3]:
import random
import torch
from eval import eval_gsm8k

# Reproducibility (optional)
random.seed(0)
torch.manual_seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)

helper = (
    "<think> Solve step by step and show your reasoning. </think>\n"
    "Then give ONLY the final integer inside <answer>...</answer>. "
    "No decimals, no extra text. </think>\n"
    "<answer>"
)

# Run a very small eval first to confirm everything works
metrics = eval_gsm8k.evaluate_gsm8k_maj16(
    model=model,
    tokenizer=tokenizer,
    n_examples=10,
    batch_size=2,
    maj_k=16,          # choose k for maj@k
    pass_k=16,          # choose k for pass@k
    temperature=0.7,   # chosse temp=0.7 for maj@16 and temp=0 for pass@1
    max_new_tokens=192,
    helper="<think>",
    prompt_style="deepseek",  # choose "base" or "deepseek"
    show_examples=2,
)

print("GSM8K smoke test metrics:")
print(metrics)



ImportError: cannot import name 'generate_completions' from 'grpo.functions' (/Users/davidhellstrom/Documents/uni/master/tdde09_projekt/reasoning-llm/grpo/functions.py)

---
## GSM8K SFT — Supervised Fine-Tuning on Real Math Problems

We now go beyond the small synthetic dataset and fine-tune on the full GSM8K training set.
Each GSM8K example already contains a step-by-step solution; we simply reformat it as
`<think>chain-of-thought</think><answer>final_answer</answer>` and train the model to
produce this exact structure.

The key differences from the earlier random-format SFT:
- **Real reasoning traces** from GSM8K rather than synthetic arithmetic steps
- **Full model fine-tuning** (all parameters, not just embeddings)
- **`SFTTrainer`** class that mirrors the `GRPOTrainer` API for consistency

### 1. Build the GSM8K SFT dataset

`GSM8KSFTDataset` tokenizes each example as:
- **prompt** (labels = -100) → the question wrapped in the DeepSeek-R1 template
- **completion** (labels = token ids) → `<think>{cot}</think><answer>{answer}</answer>`

Examples longer than `max_length` tokens are dropped to keep memory usage predictable.

In [2]:
from data.gsm8k import GSM8KSFTDataset

train_sft = GSM8KSFTDataset(
    tokenizer=tokenizer,
    prompt_template=generate_prompt,
    split="train",
    max_length=512,
)

# Inspect one example
sample = train_sft[0]
full_text = tokenizer.decode(sample["input_ids"])
print("--- Full tokenized example ---")
print(full_text)
print(f"\nInput length : {sample['input_ids'].shape[0]} tokens")
print(f"Label -100s  : {(sample['labels'] == -100).sum().item()} (prompt tokens masked)")

NameError: name 'tokenizer' is not defined

### 2. Train with `SFTTrainer`

`SFTTrainer` handles:
- Padding batches to the same length
- Standard cross-entropy loss on completion tokens only
- Periodic evaluation (loss on a held-out subset)
- Saving the best checkpoint

In [ ]:
from training.sft import SFTTrainer
from training.configs import SFTConfig

config = SFTConfig(
    lr=2e-5,
    epochs=3,
    batch_size=8,
    max_length=512,
    grad_clip=1.0,
    eval_every=200,
    eval_samples=100,
)

# Optional: load a separate eval split (raw GSM8KSFTDataset on the test set)
eval_sft = GSM8KSFTDataset(
    tokenizer=tokenizer,
    prompt_template=generate_prompt,
    split="test",
    max_length=512,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, config=config)
trainer.train(train_sft, eval_dataset=eval_sft, run_name="Qwen/Qwen3-1.7B-Base-gsm8k-sft")

### 3. Save and evaluate

After training we save the model and run the format-quality evaluation
used earlier in the notebook to confirm the model now reliably produces
`<think>` / `<answer>` blocks.

In [ ]:
from utils.models import save_model

save_model(model, tokenizer, "Qwen/Qwen3-1.7B-Base-gsm8k-sft")

In [ ]:
# Check that the model now uses the think/answer format consistently
questions = [
    "What is 11 * 9?",
    "Sam has 4 apples, eats 2, and gives 1 to Anna. How many does he have left?",
    "Solve: (5 + 1) * 21",
    "What is the next number in the sequence [1, 2, 4, 8, 16]?",
]
answers = ["99", "1", "126", "32"]
evaluate(model=model, questions=questions, answers=answers, G=16, temperature=1.0)